# Can a balanced model turn a shopper's requirements into a reliable, structured gear comparison?

## 1. Before You Begin

The model should not memorize our store catalog. Our application owns those
facts. This notebook focuses on exactly one capability: **function calling**
through the Responses API.

Deploy `gpt-5.6-terra` in Microsoft Foundry, configure section 2, and review the
[repository quickstart](../../quickstart/README.md). Current rates are on the
[Azure OpenAI pricing page](https://azure.microsoft.com/pricing/details/azure-openai/).


In [ ]:
# 2. Verify your environment
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


def find_assets() -> Path:
    """Find the shared data whether Jupyter starts here or at the repo root."""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "models/azure-openai/shared/contoso-outdoors"
        if (candidate / "products.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find models/azure-openai/shared/contoso-outdoors. "
        "Run this notebook from a checkout of the model-releases repository."
    )


required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_GPT_56_TERRA_DEPLOYMENT"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        f"Missing {missing}. Copy scripts/sample.env to .env, add the values, "
        "and review models/quickstart/README.md."
    )

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
base_url = endpoint if endpoint.endswith("/openai/v1") else f"{endpoint}/openai/v1"
client = OpenAI(api_key=os.environ["AZURE_OPENAI_API_KEY"], base_url=f"{base_url}/")
deployment = os.environ["AZURE_OPENAI_GPT_56_TERRA_DEPLOYMENT"]
assets = find_assets()

expected_assets = [
    assets / "products.json",
    assets / "manuals/product_info_1.md",
    assets / "manuals/product_info_2.md",
    assets / "images/product_1.webp",
    assets / "images/product_2.webp",
]
missing_assets = [str(path) for path in expected_assets if not path.is_file()]
if missing_assets:
    raise FileNotFoundError(f"Shared Contoso Outdoors files are missing: {missing_assets}")

print(f"Environment ready for deployment: {deployment}")
print(f"Shared assets: {assets}")


## 3. Keep catalog facts in the application

We load the shared subset and display its images for context. The model receives
only a tool contract; Python performs every product lookup and returns the
authoritative local record.


In [ ]:
# 4. Load the local product evidence
import base64
import json

from IPython.display import HTML, display

# These are the only product records used in this phase.
products = json.loads((assets / "products.json").read_text(encoding="utf-8"))
manuals = {
    product["id"]: (assets / product["manual"]).read_text(encoding="utf-8")
    for product in products
}

def display_webp(path: Path, width: int = 240) -> None:
    """Render a local WebP through HTML because IPython Image cannot embed it."""
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<img src="data:image/webp;base64,{encoded}" width="{width}">'))


for product in products:
    print(f'{product["id"]}: {product["name"]} (${product["price"]})')
    display_webp(assets / product["images"][0])

assert [product["id"] for product in products] == [1, 2]


## 5. Complete the tool loop

The shopper needs a side-by-side comparison. We require one lookup per product,
execute only recognized calls, and feed the JSON results back to the same
response chain.


In [ ]:
# 6. Let the model call the local catalog function
def lookup_product(product_id: int) -> dict:
    """Return one local product record by its stable ID."""
    match = next((product for product in products if product["id"] == product_id), None)
    if match is None:
        return {"error": f"Unknown product ID: {product_id}"}
    return match


tools = [
    {
        "type": "function",
        "name": "lookup_product",
        "description": "Look up an authoritative Contoso Outdoors product record.",
        "parameters": {
            "type": "object",
            "properties": {"product_id": {"type": "integer", "enum": [1, 2]}},
            "required": ["product_id"],
            "additionalProperties": False,
        },
        "strict": True,
    }
]
request = """
Compare product IDs 1 and 2 for a shopper preparing for a weekend camping trip.
Call lookup_product once for each product before answering. Return a concise
table with product ID, purpose, price, and one limitation. Do not invent facts.
"""

response = client.responses.create(
    model=deployment,
    input=request,
    tools=tools,
    reasoning={"effort": "medium"},
)
seen_ids = set()

# Continue until the model returns text or the bounded tool loop is exhausted.
for _ in range(3):
    calls = [item for item in response.output if item.type == "function_call"]
    if not calls:
        break
    tool_outputs = []
    for call in calls:
        if call.name != "lookup_product":
            raise ValueError(f"Unexpected tool call: {call.name}")
        arguments = json.loads(call.arguments)
        product_id = arguments["product_id"]
        seen_ids.add(product_id)
        tool_outputs.append(
            {
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(lookup_product(product_id)),
            }
        )
    response = client.responses.create(
        model=deployment,
        previous_response_id=response.id,
        input=tool_outputs,
        tools=tools,
        reasoning={"effort": "medium"},
    )
else:
    raise RuntimeError("Tool loop exceeded three Responses API turns")

assert seen_ids == {1, 2}, f"Expected lookups for IDs 1 and 2; received {seen_ids}"
assert response.output_text, "The tool loop ended without a final comparison"
print(response.output_text)


## 7. Your Turn to Explore

- Remove one ID from the tool enum and observe how the model handles the limit.
- Add a manual lookup tool without merging it into the product tool.
- Ask for a recommendation only after both tool outputs have arrived.


## 8. Summary

We used GPT-5.6 Terra for one capability: calling application-owned functions.
The model selected and parameterized the lookup; our Python code controlled
execution and supplied the facts. Review the
[function-calling primer](../../../docs/primers/function-calling.md) before
adding tools that change state or require user confirmation.


## 9. References

- [GPT-5.6 Terra model card](https://ai.azure.com/catalog/models/gpt-5.6-terra) — model positioning.
- [Use the Azure OpenAI Responses API](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses) — function tools and response chaining.
- [Azure OpenAI reasoning models](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning) — reasoning-model API behavior.
- [Foundry Models sold by Azure](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/concepts/models-sold-directly-by-azure#gpt-56) — verified model version.
